# Classification metrics (precision, recall, F1, ROC/AUC, PR)

Classification metrics ask different operational questions about the same confusion counts.

The same classifier can look good or bad depending on whether false alarms, misses, threshold ranking, or rare positives matter most. This notebook computes the count formulas once, then follows macro-F1 up the same ladder. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    cancer = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", cancer.data, cancer.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2000)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def split_scaled(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=3, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_logistic(x_tr, y_tr, C=1.0):
    model = LogisticRegression(C=C, max_iter=2500)
    model.fit(x_tr, y_tr)
    return model


def classification_error(y_true, y_pred):
    return float(1.0 - accuracy_score(y_true, y_pred))


def lesson_score(losses, cost, alternative):
    raw = round(float(np.mean(losses)), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    relative_gap = gap / alternative
    stabilized = 0.8 * score
    return {
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stabilized": stabilized,
    }


def preview_ladder(rungs):
    for name, X, y in rungs:
        labels, counts = np.unique(y, return_counts=True)
        info = dict(zip(labels.tolist(), counts.tolist()))
        print(f"{name}: X={X.shape}, classes={info}, sample={np.round(X[:2], 3).tolist()}")


## The concept, built once on D1

The lesson formula is $$precision=\frac{TP}{TP+FP},\quad recall=\frac{TP}{TP+FN},\quad F1=\frac{2PR}{P+R}$$. We first rebuild the exact lesson arithmetic before using a reusable method on larger data.

In [ ]:

def classification_metrics_precision_recall_f1_method(losses, cost, alternative):
    """Recompute the lesson raw average, cost-aware score, and alternative gap."""
    losses = np.asarray(losses, dtype=float)
    summary = lesson_score(losses, cost, alternative)
    return summary


losses = np.array([0.202, 0.148, 0.471], dtype=float)
summary = classification_metrics_precision_recall_f1_method(losses, cost=0.070, alternative=0.392)

assert abs(summary["raw"] - 0.274) < 1e-12
assert abs(summary["score"] - 0.344) < 1e-12
assert abs(summary["gap"] - 0.048) < 1e-12
assert abs(summary["relative_gap"] - 0.122) < 0.001
assert abs(summary["stabilized"] - 0.275) < 0.001

print("losses:", losses.tolist())
print("raw average:", round(summary["raw"], 3))
print("score with cost:", round(summary["score"], 3))
print("gap to alternative:", round(summary["gap"], 3))
print("relative gap:", round(summary["relative_gap"], 3))


The same score object also carries the stabilized launch number and, for this topic, the topic-specific metric calculation used on the toy D1 counts or residuals.

In [ ]:

def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2.0 * precision * recall / (precision + recall)
    return precision, recall, f1

precision, recall, f1 = precision_recall_f1(tp=2, fp=1, fn=1)
assert abs(precision - (2.0 / 3.0)) < 1e-12
assert abs(recall - (2.0 / 3.0)) < 1e-12
assert abs(f1 - (2.0 / 3.0)) < 1e-12


print("toy precision:", round(precision, 3))
print("toy recall:", round(recall, 3))
print("toy F1:", round(f1, 3))


## The dataset ladder

The notebook embeds the shared `clf_ladder()` source so it is self-contained in Colab. D1 is inspectable; D5 is real, higher-dimensional, and imbalanced enough to make the checks matter.

In [ ]:

rungs = clf_ladder()
preview_ladder(rungs)


## Run macro-F1 across D1–D5

Accuracy is printed with the shared `clf_accuracy()` helper, but macro-F1 is the tracked metric because it gives every class equal weight.

In [ ]:

def macro_f1_rung(X, y):
    x_tr, x_te, y_tr, y_te = split_scaled(X, y)
    model = fit_logistic(x_tr, y_tr)
    preds = model.predict(x_te)
    return float(f1_score(y_te, preds, average="macro"))

results = []
for name, X, y in rungs:
    metric = macro_f1_rung(X, y)
    acc = clf_accuracy(logistic_baseline, X, y)
    results.append({"name": name, "macro_f1": metric, "accuracy": acc})
    print(f"{name}: macro_F1={metric:.3f}, clf_accuracy={acc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, (name, X, y), row in zip(axes[0], rungs, results):
    x_tr, x_te, y_tr, y_te = split_scaled(X, y)
    model = fit_logistic(x_tr, y_tr)
    probs = model.predict_proba(x_te)
    confidence = np.max(probs, axis=1)
    ax.hist(confidence, bins=np.linspace(0, 1, 8), color="steelblue", edgecolor="white")
    ax.set_title(name.split(" (")[0], fontsize=8)
    ax.set_xlim(0, 1)
axes[1, 0].plot(range(1, 6), [row["macro_f1"] for row in results], marker="o")
axes[1, 0].set_title("macro-F1 vs rung")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("macro-F1")
axes[1, 0].set_ylim(0, 1.05)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

A threshold or hyperparameter can improve a raw count metric while adding instability. We keep the lesson's cost in the final choice.

In [ ]:

name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = split_scaled(X, y)
rows = []
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    model = fit_logistic(x_tr, y_tr, C=C)
    preds = model.predict(x_te)
    raw_loss = classification_error(y_te, preds)
    complexity_cost = 0.070 * math.log10(C + 1.0) / math.log10(11.0)
    decision_score = raw_loss + complexity_cost
    rows.append((C, raw_loss, complexity_cost, decision_score))

raw_best = min(rows, key=lambda row: row[1])
fixed_best = min(rows, key=lambda row: row[3])
print("D5 candidate table: C, raw validation loss, cost, decision score")
for row in rows:
    print(tuple(round(value, 4) for value in row))
print("wrong raw-only choice:", raw_best)
print("cost-aware choice:", fixed_best)
print("decision-score improvement:", round(raw_best[3] - fixed_best[3], 4))
assert fixed_best[3] <= raw_best[3] + 1e-12


## Evaluate it + Practice

- Track `macro-F1` against a no-skill baseline before trusting the result.
- Sanity-check that D1 reproduces the lesson numbers exactly and that D5 is not a toy shortcut.
- Ablation: choose the threshold by accuracy only; the metric should get worse or the selected model should become less stable.
- Failure signals: large validation gap, unstable threshold/criterion, or a cost-aware score that disagrees with the raw metric.
- Report both the raw metric and the cost-aware decision score when the lesson formula includes a cost.

Practice prompts:
1. Change one candidate hyperparameter and recompute the D5 table.
2. Replace the no-skill baseline and explain whether `macro-F1` moved for the right reason.
3. Add one diagnostic plot that would catch the named pitfall for classification metrics.